In [ ]:
# 필요한 라이브러리 설치 및 임포트
!pip install pandas
!pip install lxml
import pandas as pd
import os
import glob
import re

# 컬럼 안의 문자열 최대 표시 길이 설정 (None이면 제한 없음)
pd.set_option('display.max_colwidth', None)
# 출력할 컬럼 개수 제한 해제
pd.set_option('display.max_columns', None)
# 출력할 행 개수 제한 해제
pd.set_option('display.max_rows', None)


## 1. 학기별 개설교과목정보 데이터 합치기

In [ ]:
files = glob.glob("../data/*.xls")

all_data = []

for file in files:
    # HTML 테이블 읽기
    df = pd.read_html(file)[0]
    
    # 첫 번째 행을 컬럼명으로 지정
    df.columns = df.iloc[0]   # 0번 행을 컬럼명으로
    df = df.drop(index=0)     # 0번 행 삭제
    
    # 학기 정보 컬럼 추가
    semester = os.path.basename(file).replace("-sis.xls", "")
    df["개설학기정보"] = semester
    
    all_data.append(df)

# 모든 파일 병합
df_all = pd.concat(all_data, ignore_index=True)

df_all.head(3)


In [ ]:
df_all.tail(3)

## 2. 필요한 열 선택

In [ ]:
def make_major_df(df_all, codes=None, prefix=None):
    if codes is not None:
        df_major = df_all[df_all["과목번호"].isin(codes)]
    elif prefix is not None:
        df_major = df_all[df_all["과목번호"].str.startswith(prefix)]
    else:
        raise ValueError("codes 또는 prefix 중 하나를 지정해야 합니다.")

    result = []
    for code in df_major["과목번호"].unique():
        subset = df_major[df_major["과목번호"] == code]
        sems = ", ".join(sorted(subset["개설학기정보"].unique()))
        names = ", ".join(sorted(subset["과목명"].unique()))
        profs = ", ".join(sorted(subset["교수진"].dropna().unique()))
        result.append({
            "과목번호": code,
            "과목명": names,
            "교수": profs,
            "개설학기": sems
        })
    return pd.DataFrame(result)


## 3. 전공 데이터프레임 계절별로 만들기

In [ ]:
def make_major_dfs(df_all, major_name, codes=None, prefix=None):
    import builtins

    # 정리된 전공 데이터프레임 생성
    df_major = make_major_df(df_all, codes=codes, prefix=prefix)
    builtins.__dict__[f"df_{major_name}"] = df_major

    # 계절별 데이터 생성
    builtins.__dict__[f"df_{major_name}_spring"] = df_major[df_major["개설학기"].str.contains("-1", na=False)]
    builtins.__dict__[f"df_{major_name}_fall"] = df_major[df_major["개설학기"].str.contains("-2", na=False)]
    builtins.__dict__[f"df_{major_name}_summer"] = df_major[df_major["개설학기"].str.contains("-summer", case=False, na=False)]
    builtins.__dict__[f"df_{major_name}_winter"] = df_major[df_major["개설학기"].str.contains("-winter", case=False, na=False)]


## 4. 계절별 전공 개설학기 보기

### 4-1. 연계전공

#### 4-1-1. 빅데이터사이언스 연계전공

In [ ]:
bds_codes = [
    "STS2011", "MAT2110", "MAT3020", "MGT2002", "ECO2004", 
    "BDS4010", "CSW4010", "CSE4187",
    "AIC4012", "BDS3010", "BDS3020", "CSE4130", "CSW2010", "CSW2020",
    "CSW2030", "CSE3080", "CSW2050", "CSW3010", "CSE3081", "CSW3030", "CSE4110",
    "CSW3060", "CSW3080",
    "CSW4020", "ECO2009", "ECO3022", "ECO3023", "ECO4003", "ECO4004",
    "ECO4032", "EEE4178", "JAS4014", "MAS1004", "MAS2009",
    "MAS2010", "MAT3110", "MAT4331", "MGT4202", "MGT4208", "MGT4226",
    "MGT4515", "MGT4517", "MGT6613", "MGTG613"
]

make_major_dfs(df_all, major_name="bds", codes=bds_codes)

In [ ]:
df_bds_spring
#df_bds_summer
# df_bds_fall
#df_bds_winter

#### 4-1-2. 공공인재 연계전공

In [ ]:
pub_codes = [
    "PUB2005", "POL3130", "POL2002", "SOC2001", "SOC2003", "SOC3010",
    "ECO2001", "ECO2002", "ECO3009", "ECO3011", "ECO3017", "MGT2002", 
    "MGT2003", "PHI2005", "PSY2001", "PSY3009", "PUB3030", "PUB3016",
    "PUB3029", "PUB3023", "PUB3024", "PUB3025", "PUB3031", "PUB3032",
    "PUB3026", "PUB3021", "PUB3020", "PUB3022", "PUB3028", "PUB3027",
    "PUB4009", "KOR4500", "EDU2001", "PHI4010", "ECO2007", "MGT3004",
    "MGT4301", "MGT3005", "MGT4404"
]

make_major_dfs(df_all, major_name="pub", codes=pub_codes)


In [ ]:
df_pub_spring
#df_pub_summer
#df_pub_fall
#df_pub_winter

#### 4-1-3. 교직

In [ ]:
edu_codes = [
    "EDU2001", "EDU2002", "EDU2003", "EDU2004", "EDU3001", "EDU3047",
    "EDU3002", "EDU3033", "EDU3045", "EDU3046", "EDU3037", "EDU3004", "EDU3035",
    "EDU2005", "EDU2007", "EDU3038", "EDU3039", "EDU3048", "EDU3049",
    "SHU4019", "SHU4022", "SHU4031", "EDU3036"
]

make_major_dfs(df_all, major_name="edu", codes=edu_codes)

In [ ]:
df_edu_spring
#df_edu_summer
#df_edu_fall
#df_edu_winter

#### 4-1-4. 스포츠미디어

In [ ]:
spm_codes = [
    # 커뮤니케이션 분야
    "MAE3014", "MAS2004", "MAE3001", "MAS3001", 
    "JAS3001", "JAS3012", "JAS3005", "JAS4015", "JAS4014", "JAS3002", 
    "JAS3007", "JAS2008", "JAS4011", "JAS3010", 
    "MAE2001", "MAE3002", "MAE3003", "MAE3009", "MAE3035",
    # 스포츠 이론 분야
    "SPM3001", "SPM3004", "SPM3005", "SPM3006", "SPM3007", "SPM3008", "SPM3009", "SPM3010", "SPM3010", "SPM3011", "SPM3012", "SPM3013", "SPM3014", "SPM3015",
    # 스포츠 실기 분야
    "SPM3101", "SPM3102", "SPM3103", "SPM3104", "SPM3105", "SPM3106", "SPM3107", "SPM3108", "SPM3110", "SPM3111", "SPM3112", "SPM3113", "SPM3114", "SPM3115", "SPM3116", "SPM3117", "SPM3118", "SPM3119", "SPM3120",
]

make_major_dfs(df_all, major_name="spm", codes=spm_codes)

In [ ]:
df_spm_spring
#df_spm_summer
#df_spm_fall
#df_spm_winter

### 4-2. 일반 전공

#### 4-2-1. 경영

In [ ]:
make_major_dfs(df_all, major_name="mgt", prefix="MGT")

In [ ]:
df_mgt_spring
#df_mgt_summer
#df_mgt_fall
#df_mgt_winter

#### 4-2-2. 국문

In [ ]:
make_major_dfs(df_all, major_name="kor", prefix="KOR")

In [ ]:
df_kor_spring
#df_kor_summer
#df_kor_fall
#df_kor_winter

#### 4-2-3. 경제

In [ ]:
make_major_dfs(df_all, major_name="eco", prefix="ECO")

In [ ]:
df_eco_spring
#df_eco_summer
#df_eco_fall
#df_eco_winter

#### 4-2-4. 수학

In [ ]:
make_major_dfs(df_all, major_name="mat", prefix="MAT")

In [ ]:
df_mat_spring
#df_mat_summer
#df_mat_fall
#df_mat_winter